Build Your Personalized Knowledge Base: Take your college roll number. Extract its digits. Build a pandas
DataFrame with exactly 6 FAQ entries: 4 fixed entries (given in table below) and other 2 entries constructed from your
own roll number digits as follows:
• Take the LAST TWO DIGITS of your roll number. For each digit d, compute category = ["billing", "account",
"general"][d % 3]. Invent one realistic question+answer+3 keywords per entry that fits the assigned category (e.g.
if d%3 gives "account", write a question like “how do I update my registered mobile number”).
• # Example roll number ...23 -> digits 2, 3
• # digit 2 -> category[2 % 3] = general
• # digit 3 -> category[3 % 3] = billing

In [12]:
import pandas as pd

roll_no = 102417426

fixed_entries = [
    {"question": "what is the annual fee", "answer": "The annual fee is Rs 500.", "keywords": "fee cost price charge", "category": "billing"},
    {"question": "how to reset password", "answer": "Go to Settings > Reset Password.", "keywords": "password reset login", "category": "account"},
    {"question": "what are your working hours", "answer": "We are open 9 AM to 5 PM.", "keywords": "hours timing open time", "category": "general"},
    {"question": "how can i pay the fee", "answer": "You can pay via UPI, card, or net banking.", "keywords": "pay payment upi fee", "category": "billing"}
]

last_two_digits = [2, 6]
categories = ["billing", "account", "general"]

personalized_entries = []

for d in last_two_digits:
    category = categories[d % 3]

    if category == "general":
        personalized_entries.append({
            "question": "how can i contact customer support",
            "answer": "You can contact customer support during working hours.",
            "keywords": "support help contact",
            "category": category
        })
    elif category == "billing":
        personalized_entries.append({
            "question": "how can i check my payment status",
            "answer": "You can check your payment status from the billing section.",
            "keywords": "payment status billing",
            "category": category
        })

df = pd.DataFrame(fixed_entries + personalized_entries)

print(df)

                             question  \
0              what is the annual fee   
1               how to reset password   
2         what are your working hours   
3               how can i pay the fee   
4  how can i contact customer support   
5   how can i check my payment status   

                                              answer                keywords  \
0                          The annual fee is Rs 500.   fee cost price charge   
1                   Go to Settings > Reset Password.    password reset login   
2                          We are open 9 AM to 5 PM.  hours timing open time   
3         You can pay via UPI, card, or net banking.     pay payment upi fee   
4  You can contact customer support during workin...    support help contact   
5  You can check your payment status from the bil...  payment status billing   

  category  
0  billing  
1  account  
2  general  
3  billing  
4  general  
5  billing  


Generate and Score a Hypothesis - Implement a scoring function that takes a query string and returns all matching
entries ranked by confidence

In [13]:
def score_query(query, df):
    query_words = set(query.lower().split())
    results = []

    for index, row in df.iterrows():
        keyword_words = set(row["keywords"].lower().split())
        score = len(query_words & keyword_words)

        if score > 0:
            results.append((index, score, row["question"]))

    results.sort(key=lambda x: x[1], reverse=True)

    return results

query = input("Enter your query: ")
results = score_query(query, df)

print("\nQ2:")
for index, score, question in results:
    print("Confidence:", score, "|", question)

Enter your query: fee

Q2:
Confidence: 1 | what is the annual fee
Confidence: 1 | how can i pay the fee


Write a function same_category(category_name, df) that returns all questions belonging to a given category. Call it
using the category of one of the personalized entries from Q1, and print the result.

In [14]:
def same_category(category_name, df):
    return df[df["category"] == category_name]

category_name = df.iloc[4]["category"]

print("\nQ3:")
print(same_category(category_name, df))


Q3:
                             question  \
2         what are your working hours   
4  how can i contact customer support   

                                              answer                keywords  \
2                          We are open 9 AM to 5 PM.  hours timing open time   
4  You can contact customer support during workin...    support help contact   

  category  
2  general  
4  general  


Pick any one entry in your knowledge base. Ask the user to input a new keyword, add it to that entry's keywords, and
save your entire updated DataFrame to a CSV file named <your_roll_number>_faq_data.csv.

In [11]:
index = 0

new_keyword = input("Enter a new keyword: ")

df.loc[index, "keywords"] = df.loc[index, "keywords"] + " " + new_keyword

filename = str(roll_no) + "_faq_data.csv"

df.to_csv(filename, index=False)

print("\nQ4:")

print(df)

print("File saved as:", filename)

Enter a new keyword: yearly

Q4:
                             question  \
0              what is the annual fee   
1               how to reset password   
2         what are your working hours   
3               how can i pay the fee   
4  how can i contact customer support   
5   how can i check my payment status   

                                              answer  \
0                          The annual fee is Rs 500.   
1                   Go to Settings > Reset Password.   
2                          We are open 9 AM to 5 PM.   
3         You can pay via UPI, card, or net banking.   
4  You can contact customer support during workin...   
5  You can check your payment status from the bil...   

                       keywords category  
0  fee cost price charge yearly  billing  
1          password reset login  account  
2        hours timing open time  general  
3           pay payment upi fee  billing  
4          support help contact  general  
5        payment status bill

Using groupby, print how many FAQ entries you have per category.

In [7]:
print("\nQ5:")
print(df.groupby("category").size())


Q5:
category
account    1
billing    3
general    2
dtype: int64


Modify your Q2 scoring function so that if two or more entries tie for the highest score, it does not silently pick one
— it prints all matching entries instead, so the user can see every equally good match. Demonstrate with one query that
produces a tie (e.g. a query matching both "fee" entries) and one that doesn't.

In [8]:
def score_query_with_ties(query, df):
    query_words = set(query.lower().split())
    results = []

    for index, row in df.iterrows():
        keyword_words = set(row["keywords"].lower().split())
        score = len(query_words & keyword_words)

        if score > 0:
            results.append((index, score, row["question"]))

    if not results:
        print("No matching entries found.")
        return

    highest_score = max(result[1] for result in results)

    matches = [result for result in results if result[1] == highest_score]

    print("Highest confidence:", highest_score)

    for index, score, question in matches:
        print(question)

print("\nQ6 Tie:")
score_query_with_ties("fee", df)

print("\nQ6 No Tie:")
score_query_with_ties("password", df)


Q6 Tie:
Highest confidence: 1
what is the annual fee
how can i pay the fee

Q6 No Tie:
Highest confidence: 1
how to reset password
